In [147]:

%pip install --upgrade pandas numpy seaborn plotly 


[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Business Case: Kiva Lending Patterns
## An Exploratory Data Analysis and Interactive Dashboard

### Project Objective

The objective of this project is to explore Kiva's historical
lending data, identify patterns in loan distribution across
sectors and countries, and investigate differences between
requested and funded loan amounts.

The project includes data inspection and cleaning, exploratory
data visualization, a funding comparison, and an interactive
dashboard developed using Dash.

In [148]:
# Importing necessary libraries for EDA

# Data manipulation and numerical operations
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly as py

# Operating system interactions
import os

### 1. Loading the Dataset

The file uses `#` as the column separator. The first column in the file is an unnamed row index, so `index_col=0` is used to make it the DataFrame index. `skipinitialspace=True` strips the leading space that appears after each `#` in the header, giving clean column names without further post-processing.

In [149]:
# Load the dataset
DATA_PATH = os.path.join('..', 'data', 'data_abschlussprojekt.csv')

df = pd.read_csv(
    DATA_PATH,
    sep='#',
    index_col=0,
    skipinitialspace=True
)

# print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head(5)

,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,term_in_months,lender_count,borrower_genders,repayment_interval
0,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,12.0,12,female,irregular
1,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,11.0,14,"female, female",irregular
2,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,43.0,6,female,bullet
3,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,11.0,8,female,irregular
4,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,14.0,16,female,monthly


### 2. Initial Inspection

Before any cleaning, inspect the raw state of the data: column names, data types, first rows, summary statistics, missing values, and duplicates.

In [150]:
# Display the shape of the DataFrame
df.shape

(671205, 13)

In [151]:
# Display the column names of the DataFrame
df.columns.to_list()  # Convert the column names to a list for easier inspection

['funded_amount',
 'loan_amount',
 'activity',
 'sector',
 'use',
 'country_code',
 'country',
 'region',
 'currency',
 'term_in_months',
 'lender_count',
 'borrower_genders',
 'repayment_interval']

In [152]:
# Display the data types of each column in the DataFrame
df.dtypes

funded_amount         float64
loan_amount           float64
activity                  str
sector                    str
use                       str
country_code              str
country                   str
region                    str
currency                  str
term_in_months        float64
lender_count            int64
borrower_genders          str
repayment_interval        str
dtype: object

In [153]:
# Display concise summary of the DataFrame, including column data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 671205 entries, 0 to 671204
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   funded_amount       671205 non-null  float64
 1   loan_amount         671205 non-null  float64
 2   activity            671205 non-null  str    
 3   sector              671205 non-null  str    
 4   use                 666972 non-null  str    
 5   country_code        671197 non-null  str    
 6   country             671205 non-null  str    
 7   region              614405 non-null  str    
 8   currency            671205 non-null  str    
 9   term_in_months      671205 non-null  float64
 10  lender_count        671205 non-null  int64  
 11  borrower_genders    666984 non-null  str    
 12  repayment_interval  671205 non-null  str    
dtypes: float64(3), int64(1), str(9)
memory usage: 148.1 MB


In [154]:
# Display summary statistics for all columns, including categorical and numerical data
df.describe(include='all')

,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,term_in_months,lender_count,borrower_genders,repayment_interval
count,671205.000000,671205.000000,671205,671205,666972,671197,671205,614405,671205,671205.000000,671205.000000,666984,671205
unique,NaN,NaN,163,15,423452,86,87,12695,67,NaN,NaN,11298,4
top,NaN,NaN,Farming,Agriculture,to buy a water filter to provide safe drinking...,PH,Philippines,Kaduna,PHP,NaN,NaN,female,monthly
freq,NaN,NaN,72955,180302,5217,160441,160441,10000,160440,NaN,NaN,426502,342717
mean,785.995061,842.397107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.739022,20.590922,NaN,NaN
std,1130.398941,1198.660073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.598919,28.459551,NaN,NaN
min,0.000000,25.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,NaN
25%,250.000000,275.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.000000,7.000000,NaN,NaN
50%,450.000000,500.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.000000,13.000000,NaN,NaN
75%,900.000000,1000.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.000000,24.000000,NaN,NaN


#### Missing Values

Count and percentage of missing values per column.

In [155]:
# Calculate and display missing values for each column in the DataFrame
# Calculate missing counts and percentages
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

# Combine into a new DataFrame
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})

# Filter to show only columns that have at least one missing value
missing_df = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)

# Display the result
print("Columns with missing values:")
# print(missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False))
display(missing_df)


Columns with missing values:


,Missing Count,Missing %
region,56800,8.46
use,4233,0.63
borrower_genders,4221,0.63
country_code,8,0.00


#### Duplicate Rows

In [156]:
# Display the count of unique values for each column
df.nunique()

funded_amount            610
loan_amount              479
activity                 163
sector                    15
use                   423452
country_code              86
country                   87
region                 12695
currency                  67
term_in_months           148
lender_count             503
borrower_genders       11298
repayment_interval         4
dtype: int64

In [157]:
# Display all duplicated rows in the DataFrame
df[df.duplicated(keep=False)]

,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,term_in_months,lender_count,borrower_genders,repayment_interval
327,275.0,275.0,Farming,Agriculture,to buy fertilizers and other farm supplies.,PH,Philippines,"Brookes Point, Palawan",PHP,8.0,8,female,irregular
392,100.0,100.0,Home Energy,Personal Use,to buy a solar lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
405,100.0,100.0,Home Energy,Personal Use,to buy a solar-powered lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
498,100.0,100.0,Home Energy,Personal Use,to buy a solar-powered lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
606,100.0,100.0,Home Energy,Personal Use,to buy a solar-powered lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
...,...,...,...,...,...,...,...,...,...,...,...,...,...
671200,0.0,25.0,Livestock,Agriculture,"[True, u'para compara: cemento, arenya y ladri...",PY,Paraguay,Concepción,USD,13.0,0,female,monthly
671201,25.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,13.0,1,female,monthly
671202,0.0,25.0,Games,Entertainment,NaN,KE,Kenya,NaN,KES,13.0,0,NaN,monthly
671203,0.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,13.0,0,female,monthly


In [158]:
# Check for exact duplicate rows in the DataFrame
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes:,}")  
print(f"Unique rows: {len(df) - n_dupes:,} of {len(df):,} ({(1 - n_dupes/len(df))*100:.2f}%)")  # Print the number and percentage of unique rows

Exact duplicate rows: 24,588
Unique rows: 646,617 of 671,205 (96.34%)


#### Categorical Column Values

Check unique values in key categorical columns to catch encoding inconsistencies before cleaning.

In [159]:
print("Unique sectors:", sorted(df['sector'].dropna().unique()))
print("\nUnique repayment intervals:", sorted(df['repayment_interval'].dropna().unique()))

Unique sectors: ['Agriculture', 'Arts', 'Clothing', 'Construction', 'Education', 'Entertainment', 'Food', 'Health', 'Housing', 'Manufacturing', 'Personal Use', 'Retail', 'Services', 'Transportation', 'Wholesale']

Unique repayment intervals: ['bullet', 'irregular', 'monthly', 'weekly']


#### Numerical Column Ranges

Check for negative or zero values in columns that should be strictly positive.

In [160]:
for col in ['funded_amount', 'loan_amount', 'term_in_months', 'lender_count']:
    neg = (df[col] < 0).sum()
    zero = (df[col] == 0).sum()
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, zeros={zero:,}, negatives={neg:,}")

funded_amount: min=0.0, max=100000.0, zeros=3,383, negatives=0
loan_amount: min=25.0, max=100000.0, zeros=0, negatives=0
term_in_months: min=1.0, max=158.0, zeros=0, negatives=0
lender_count: min=0, max=2986, zeros=3,383, negatives=0


In [161]:
# Create a data dictionary containing column names, data types, missing values, and unique values
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "dtype": df.dtypes.values,
    "Data type": [
        type(df[col].iloc[0]).__name__
        for col in df.columns
    ],
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
    
})
data_dictionary

,Column,dtype,Data type,Missing Values,Unique Values
0,funded_amount,float64,float64,0,610
1,loan_amount,float64,float64,0,479
2,activity,str,str,0,163
3,sector,str,str,0,15
4,use,str,str,4233,423452
5,country_code,str,str,8,86
6,country,str,str,0,87
7,region,str,str,56800,12695
8,currency,str,str,0,67
9,term_in_months,float64,float64,0,148


### 3. Data Cleaning

The following cleaning steps are applied in sequence. Each step documents the decision and its justification.

#### Step 1 - Strip Whitespace from String Columns

String values in the raw file may carry leading or trailing whitespace. This is stripped uniformly across all object columns without losing any data.

In [162]:
str_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip()
print(f"Stripped whitespace from {len(str_cols)} string columns: {str_cols}")

Stripped whitespace from 9 string columns: ['activity', 'sector', 'use', 'country_code', 'country', 'region', 'currency', 'borrower_genders', 'repayment_interval']


#### Step 2 - Duplicate Rows

Exact duplicate rows are dropped only if confirmed. A row is an exact duplicate if every column value matches another row. Kiva loan records are not expected to repeat; genuine duplicates are data entry errors.

In [163]:
n_before = len(df)
df = df.drop_duplicates()
n_dropped = n_before - len(df)
print(f"Rows before: {n_before:,}")
print(f"Exact duplicates dropped: {n_dropped:,}")
print(f"Rows after: {len(df):,}")

Rows before: 671,205
Exact duplicates dropped: 25,579
Rows after: 645,626


#### Step 3 - Missing Values: `region`

`region` is a free-text field describing the sub-national area within a country. Kiva data commonly has a large share of missing regions because lenders are not required to supply them. This step records the count; the fill to `'Not specified'` happens in Step 9 together with all remaining string columns.

In [164]:
missing_region = df['region'].isnull().sum()
pct_region = missing_region / len(df) * 100
print(f"Missing region: {missing_region:,} ({pct_region:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

Missing region: 56,136 (8.7% of rows)
Will be filled with 'Not specified' in Step 9.


#### Step 4 - Missing Values: `borrower_genders`

`borrower_genders` holds one or more gender labels separated by commas. Missing values indicate the information was not supplied. This step records the count; the fill happens in Step 9.

In [165]:
missing_bg = df['borrower_genders'].isnull().sum()
pct_bg = missing_bg / len(df) * 100
print(f"Missing borrower_genders: {missing_bg:,} ({pct_bg:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

Missing borrower_genders: 3,888 (0.6% of rows)
Will be filled with 'Not specified' in Step 9.


#### Step 5 - Missing Values: `use`

`use` is a free-text description of how the borrower intends to use the loan. This step records the count; the fill happens in Step 9.

In [166]:
missing_use = df['use'].isnull().sum()
pct_use = missing_use / len(df) * 100
print(f"Missing use: {missing_use:,} ({pct_use:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

Missing use: 3,900 (0.6% of rows)
Will be filled with 'Not specified' in Step 9.


#### Step 6 - Missing Values: `country_code`

Eight rows have a missing `country_code` but a populated `country` value of 'Namibia'. The ISO 3166-1 alpha-2 code for Namibia is 'NA'. These eight rows are filled rather than dropped because the country is unambiguously known.

In [167]:
missing_cc = df['country_code'].isnull().sum()
print(f"Missing country_code before fill: {missing_cc:,}")
print("Countries for these rows:")
print(df.loc[df['country_code'].isnull(), 'country'].value_counts().to_string())

df.loc[df['country_code'].isnull() & (df['country'] == 'Namibia'), 'country_code'] = 'NA'

remaining_cc = df['country_code'].isnull().sum()
print(f"\nMissing country_code after fill: {remaining_cc:,}")

Missing country_code before fill: 8
Countries for these rows:
country
Namibia    8

Missing country_code after fill: 0


#### Step 7 - Missing Values: `term_in_months`

Confirm whether `term_in_months` has any missing values. If present, they would be retained rather than imputed, since imputing duration without supporting data would distort the distribution.

In [168]:
missing_term = df['term_in_months'].isnull().sum()
pct_term = missing_term / len(df) * 100
print(f"Missing term_in_months: {missing_term:,} ({pct_term:.1f}% of rows)")

Missing term_in_months: 0 (0.0% of rows)


#### Step 8 - Remaining Missing Values Check

Confirm which columns still have NaN before the fill step.

In [169]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if remaining_missing.empty:
    print("No NaN remaining before fill step.")
else:
    print("Columns with NaN before fill:")
    for col, cnt in remaining_missing.items():
        print(f"  {col}: {cnt:,} ({cnt/len(df)*100:.2f}%)")

Columns with NaN before fill:
  use: 3,900 (0.60%)
  region: 56,136 (8.69%)
  borrower_genders: 3,888 (0.60%)


#### Step 9 - Fill Remaining String NaN with 'Not specified'

All remaining NaN values in string columns are filled with `'Not specified'`. This makes the absence of data explicit and visible in groupby operations and visualizations instead of being silently excluded. The label `'Not specified'` is chosen over `'Unknown'` to avoid implying the information exists but is unclear -- it simply was not recorded.

In [170]:
str_cols_with_nan = df.select_dtypes(include=['object', 'str']).columns[
    df.select_dtypes(include=['object', 'str']).isnull().any()
].tolist()

for col in str_cols_with_nan:
    filled = df[col].isnull().sum()
    df[col] = df[col].fillna('Not specified')
    print(f"  {col}: filled {filled:,} NaN with 'Not specified'")

print(f"\nTotal NaN remaining in string columns: {df.select_dtypes(include=['object', 'str']).isnull().sum().sum()}")

  use: filled 3,900 NaN with 'Not specified'
  region: filled 56,136 NaN with 'Not specified'
  borrower_genders: filled 3,888 NaN with 'Not specified'

Total NaN remaining in string columns: 0


#### Step 10 - Numerical Validity

Records with a `loan_amount` of zero or below are not meaningful loan records. `funded_amount` can legitimately be zero (partially funded) but not negative. Rows failing these checks are dropped with a count reported.

In [171]:
n_before = len(df)

invalid_loan = df['loan_amount'] <= 0
invalid_funded = df['funded_amount'] < 0
invalid_mask = invalid_loan | invalid_funded

print(f"Rows with loan_amount <= 0:  {invalid_loan.sum():,}")
print(f"Rows with funded_amount < 0: {invalid_funded.sum():,}")
print(f"Total invalid rows to drop:  {invalid_mask.sum():,}")

df = df[~invalid_mask].copy()
print(f"\nRows after numerical validity filter: {len(df):,} (dropped {n_before - len(df):,})")

Rows with loan_amount <= 0:  0
Rows with funded_amount < 0: 0
Total invalid rows to drop:  0

Rows after numerical validity filter: 645,626 (dropped 0)


#### Step 11 - Unusual `term_in_months` Values

Inspect the distribution of `term_in_months` to identify unusually short or long durations. Unusual values are noted but not removed unless they are clearly data entry errors (negative or zero), since genuine long-term loans exist.

In [172]:
print("term_in_months distribution:")
print(df['term_in_months'].describe())
print(f"\nRows with term_in_months <= 0: {(df['term_in_months'] <= 0).sum():,}")
print(f"Rows with term_in_months > 120 (more than 10 years): {(df['term_in_months'] > 120).sum():,}")

term_in_months distribution:
count    645626.000000
mean         13.721718
std           8.477883
min           1.000000
25%           8.000000
50%          13.000000
75%          14.000000
max         158.000000
Name: term_in_months, dtype: float64

Rows with term_in_months <= 0: 0
Rows with term_in_months > 120 (more than 10 years): 208


Drop rows where `term_in_months` is zero or negative; these cannot represent valid loan durations.

In [173]:
n_before = len(df)
invalid_term = df['term_in_months'].notna() & (df['term_in_months'] <= 0)
df = df[~invalid_term].copy()
print(f"Dropped {n_before - len(df):,} rows with term_in_months <= 0.")
print(f"Rows remaining: {len(df):,}")

Dropped 0 rows with term_in_months <= 0.
Rows remaining: 645,626


#### Step 12 - Derived Feature: `funding_gap`

`funding_gap` is the difference between the requested amount and the funded amount. A positive value means the loan was partially funded. A negative value means more was disbursed than originally requested; the assignment context confirms this can happen and these records are kept. Zero means fully funded.

In [174]:
df['funding_gap'] = df['loan_amount'] - df['funded_amount']

fully_funded = (df['funding_gap'] == 0).sum()
partially_funded = (df['funding_gap'] > 0).sum()
over_funded = (df['funding_gap'] < 0).sum()

print(f"Fully funded (gap = 0):          {fully_funded:,} ({fully_funded/len(df)*100:.1f}%)")
print(f"Partially funded (gap > 0):      {partially_funded:,} ({partially_funded/len(df)*100:.1f}%)")
print(f"Over-funded (gap < 0):           {over_funded:,} ({over_funded/len(df)*100:.1f}%)")

print("\nfunding_gap statistics:")
print(df['funding_gap'].describe())

Fully funded (gap = 0):          597,599 (92.6%)
Partially funded (gap > 0):      48,025 (7.4%)
Over-funded (gap < 0):           2 (0.0%)

funding_gap statistics:
count    645626.000000
mean         58.499557
std         399.368658
min        -400.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       50000.000000
Name: funding_gap, dtype: float64


#### Step 13 - Final Cleaned Dataset Summary

In [175]:
print("Final cleaned dataset:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print("\nRemaining missing values:")
final_missing = df.isnull().sum()
print(final_missing[final_missing > 0])
print("\nFirst rows:")
df.head()

Final cleaned dataset:
  Shape: (645626, 14)
  Columns: ['funded_amount', 'loan_amount', 'activity', 'sector', 'use', 'country_code', 'country', 'region', 'currency', 'term_in_months', 'lender_count', 'borrower_genders', 'repayment_interval', 'funding_gap']

Remaining missing values:
Series([], dtype: int64)

First rows:


,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,term_in_months,lender_count,borrower_genders,repayment_interval,funding_gap
0,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,12.0,12,female,irregular,0.0
1,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,11.0,14,"female, female",irregular,0.0
2,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,43.0,6,female,bullet,0.0
3,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,11.0,8,female,irregular,0.0
4,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,14.0,16,female,monthly,0.0


In [176]:
# Display the shape of the DataFrame after cleaning
df.shape

(645626, 14)

In [177]:
# Create a data dictionary containing column names, data types, missing values, and unique values after cleaning
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "dtype": df.dtypes.values,
    "Data type": [
        type(df[col].iloc[0]).__name__
        for col in df.columns
    ],
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
    
})
data_dictionary

,Column,dtype,Data type,Missing Values,Unique Values
0,funded_amount,float64,float64,0,610
1,loan_amount,float64,float64,0,479
2,activity,str,str,0,163
3,sector,str,str,0,15
4,use,str,str,0,417852
5,country_code,str,str,0,87
6,country,str,str,0,87
7,region,str,str,0,12696
8,currency,str,str,0,67
9,term_in_months,float64,float64,0,148


### 4. Save Cleaned Dataset

The cleaned and processed dataset is saved as a CSV to the `data` directory for use in subsequent analysis and visualization steps.

In [178]:
OUTPUT_PATH = os.path.join('..', 'data', 'kiva_cleaned.csv')
df.to_csv(OUTPUT_PATH)
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

Saved cleaned dataset to: ../data/kiva_cleaned.csv
Shape: 645,626 rows, 14 columns
